In [ ]:
import h5py, uproot
import awkward as ak, numpy as np

try:
    from IPython import get_ipython
    if 'IPKernelApp' in get_ipython().config:
        from tqdm.notebook import tqdm
    else:
        from tqdm import tqdm
except Exception:
    from tqdm import tqdm


def root_to_hdf5(root_path: str, tree_name: str, hdf5_path: str, batch_size: int = 500):
    """
    Convert a ROOT file with event-wise arrays into an HDF5 file.
    
    Each ROOT entry is assumed to have arrays:
        xx, zz, view, slice_label, cp_label

    Output HDF5 structure:
        /events/<event_id>/hits           (N_hits, 3) float32
        /events/<event_id>/slice_labels   (N_hits,)   int64
        /events/<event_id>/cp_labels      (N_hits,)   int64
    """
    # Open ROOT tree
    file = uproot.open(root_path)
    tree = file[tree_name]

    # Count entries for progress bar
    n_events = tree.num_entries

    print(f"Found {n_events} events in {tree_name} tree.")

    # Open output HDF5
    with h5py.File(hdf5_path, "w") as h5f:
        event_id = 0
        for batch_start in tqdm(range(0, n_events, batch_size), desc="Converting"):
            batch_stop = min(batch_start + batch_size, n_events)

            arrays = tree.arrays(["xx", "zz", "view", "slice_label", "cp_label"], entry_start=batch_start, entry_stop=batch_stop)

            xx_batch = arrays["xx"]
            zz_batch = arrays["zz"]
            vv_batch = arrays["view"]
            slice_batch = arrays["slice_label"]
            cp_batch = arrays["cp_label"]

            # Loop over events in batch
            for i in range(len(xx_batch)):
                xx = ak.to_numpy(xx_batch[i])
                zz = ak.to_numpy(zz_batch[i])
                vv = ak.to_numpy(vv_batch[i])
                slice_labels = ak.to_numpy(slice_batch[i])
                cp_labels = ak.to_numpy(cp_batch[i])

                views = np.unique(vv)
                for v in views:
                    event_id += 1
                    idx = np.where(vv == v)
                    # Stack hits: (N, 3)
                    hits = np.stack([xx[idx], zz[idx], vv[idx]], axis=1).astype(np.float32)
    
                    # Create HDF5 group
                    g = h5f.create_group(f"events/{event_id}")
                    g.create_dataset("hits", data=hits, compression="gzip")
                    g.create_dataset("slice_labels", data=slice_labels[idx].astype(np.int64), compression="gzip")
                    g.create_dataset("cp_labels", data=cp_labels[idx].astype(np.int64), compression="gzip")

In [ ]:
root_to_hdf5("training.root", "slices", "training.h5")